# 03 · RAG system design

**Deck section 3** · slides 21–32

Index-time and query-time are two different systems with two different SLAs. Keep them
separate in your head and in your repo. Everything above the query path is a batch job with a
deploy step; everything on the query path has a p95; and the control plane is what lets you
change either one without guessing.

**By the end you can**

- draw the four-plane reference architecture from memory and say what each plane owns
- choose a chunking strategy from the shape of the corpus and defend it with a measurement
- keep an index fresh without a nightly full rebuild — and explain the outage that happens
  when you mix the incremental and rebuild paths
- design permission-aware retrieval that survives the sentence "no answer may be *influenced*
  by a document the user cannot read"


In [ ]:
import pathlib, sys
ROOT = pathlib.Path.cwd()
while not (ROOT / "raglab" / "__init__.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from raglab.bootstrap import bootstrap
bootstrap(verbose=False)

import pandas as pd
import raglab
from raglab import (viz, tables, catalog, chunking, corpus, embed, metrics,
                     pipeline, retrieve, store, trace)
viz.reset_figures("3."); tables.reset_tables("3.")

bundle, index, pipe = raglab.quickstart(**raglab.TUNED)
eval_sample = [q for q in bundle.questions if q.question_type != "null"][:70]

---

## 3.1 The reference architecture

Four planes. This is the whiteboard you should be able to draw from memory in an interview,
and the labels matter as much as the boxes: *index path* is offline and versioned, *stores*
are the state, *query path* has a p95, and the *control plane* is the only reason you can
change either path deliberately.


In [ ]:
viz.hld([
    dict(name="Index path", note="offline · batch · versioned", tone="index", nodes=[
        ("Sources", "docs · tickets · code"),
        ("Parse & normalise", "layout · tables · OCR"),
        ("Chunk & enrich", "context · titles · ACL"),
        ("Embed", "batched · model version pinned"),
        ("Publish", "blue/green alias swap")]),
    dict(name="Stores", tone="store", chain=False, nodes=[
        ("Vector index", "HNSW / IVF-PQ"),
        ("Lexical index", "BM25 postings"),
        ("Doc & chunk store", "text · provenance"),
        ("Metadata + ACL", "filters · tenancy")]),
    dict(name="Query path", note="online · p95 · every request", tone="query", nodes=[
        "Query", "Route & rewrite", "Hybrid retrieve N≈100", "Fuse", "Rerank →25",
        "Pack k≈8", "Generate", "Verify & cite"]),
    dict(name="Control plane", note="how you change either path on purpose", tone="control",
         nodes=[("Trace store", "scores · context · latency"),
                ("Eval harness", "offline suites + judge"),
                ("Release gate", "blocks the deploy"),
                ("Feedback", "failures rejoin the eval set")]),
], title="Production RAG reference architecture", kicker="HLD · draw this from memory",
   caption="Everything above the query path is a batch job with a deploy step. Everything on "
           "the query path has a p95. The control plane is what lets you change either one "
           "without guessing.",
   source="Deck slide 22")

In [ ]:
tables.show(pd.DataFrame([
    ["Index path", "chunking.py · embed.py · store.upsert()",
     "Runs in ~0.2 s here, hours on a client corpus", "A deploy, with a rollback"],
    ["Stores", "store.InMemoryIndex — FTS5 + vector table + NSW graph + ACL column",
     "sqlite3 :memory:", "OpenSearch / pgvector / a managed vector DB"],
    ["Query path", "retrieve.py · context.py · generate.py",
     "~35 ms per query offline", "The thing with an SLA and a pager"],
    ["Control plane", "trace.py · metrics.py · judge.py · pipeline.evaluate()",
     "A second in-memory SQLite database", "CI job + trace store + a gate that can say no"],
], columns=["Plane", "Where it lives in this toolkit", "What it costs here",
            "What it is in production"]),
    title="The architecture, mapped onto code you can read",
    kicker="Orientation",
    caption="Every box in the diagram is a file. That is the point of building it small "
            "first — you can hold the whole thing before you argue about the managed version.",
    emphasize="Plane")

---

## 3.2 Index-time: the index is a product decision

Four steps, and the fourth is the one teams skip. *Validate before release*: document
coverage, duplicate chunks, and retrieval quality. The next cell runs all three against the
live index.


In [ ]:
viz.flow([
    ("Normalise", "retain source, title, date and other metadata — a field you did not index "
                  "is a filter you cannot apply"),
    ("Chunk deliberately", "choose boundaries and overlap that preserve meaning"),
    ("Build complementary indexes", "lexical for exact matching, vectors for semantic"),
    ("Validate before release", "coverage, duplicates, retrieval quality — then publish"),
], title="Index-time: prepare the knowledge base", kicker="Section 3 · index path",
   caption="The index is a product decision: its structure determines what the query path "
           "can find.", source="Deck slide 23")

In [ ]:
# ── Pre-release index validation ────────────────────────────────────────────
from raglab.embed import tokenize

indexed_docs = {r[0] for r in index.db.execute(
    "SELECT DISTINCT doc_id FROM chunks WHERE index_version='v1' AND tombstoned=0")}
missing = [d.doc_id for d in bundle.documents if d.doc_id not in indexed_docs]

sigs = {}
for c in pipe.chunks:
    key = frozenset(tokenize(c.text))
    sigs.setdefault(key, []).append(c.chunk_id)
dupes = {k: v for k, v in sigs.items() if len(v) > 1}

no_meta = [c for c in pipe.chunks if not c.published or not c.title]
mixed = index.mixed_version_check("v1")

probe = pipeline.evaluate(pipe, eval_sample[:30], pipe.chunks, personas=bundle.personas)
probe_recall = sum(r["evidence_recall_at_N"] for r in probe) / len(probe)

checks = [
    ["Document coverage", f"{len(indexed_docs)}/{len(bundle.documents)} documents indexed",
     "PASS" if not missing else f"FAIL — {len(missing)} missing",
     "A document that never reached the index is a question you can never answer."],
    ["Duplicate chunks", f"{sum(len(v) for v in dupes.values())} chunks in "
     f"{len(dupes)} duplicate groups",
     "PASS" if len(dupes) < 0.02 * len(pipe.chunks) else "WARN",
     "Duplicates are distractors that also cost tokens. Deduplicate before packing."],
    ["Metadata completeness", f"{len(pipe.chunks) - len(no_meta)}/{len(pipe.chunks)} chunks "
     "carry title and date",
     "PASS" if not no_meta else f"FAIL — {len(no_meta)} incomplete",
     "Title or date not indexed means a required filter cannot be applied — and temporal "
     "questions become unanswerable."],
    ["Encoder version consistency", f"{list(mixed['tags'])}",
     "PASS" if mixed["ok"] else "FAIL — mixed-version index",
     "Vectors from two models are not comparable, and cosine similarity will not tell you "
     "they are wrong."],
    ["Retrieval quality probe", f"Evidence Recall@N = {probe_recall:.3f} on 30 held questions",
     "PASS" if probe_recall > 0.80 else "WARN",
     "A smoke test on the eval set, run before the alias swap. Cheap, and it catches a "
     "chunking change that silently halved recall."],
]
tables.show(pd.DataFrame(checks, columns=["Check", "Measured", "Verdict", "Why it blocks a release"]),
            title="Index validation, run before publish",
            kicker="Pre-release gate",
            caption="Five checks, all cheap, all automatable. Every one of them corresponds "
                    "to a section-3 failure point.",
            emphasize="Verdict",
            highlight_rows=lambda r: not str(r["Verdict"]).startswith("PASS"))

---

## 3.3 Chunking: answer it with the shape of the corpus

Do not answer chunk size with a number. Answer it with the shape of the document and the
shape of the question — and then measure, because the tree gives you a starting point, not a
result.

### What the code is about to do

Profile the corpus to answer the tree's four questions from evidence, run the tree, then run
*all seven* strategies through the same eval set and see whether the tree's recommendation
survives contact with the measurement.


In [ ]:
# Answer the tree's questions from the corpus itself rather than from an opinion.
structured = sum(1 for d in bundle.documents if len(d.passages) > 1) / len(bundle.documents)
q_tokens = [len(tokenize(q.query)) for q in bundle.questions]
short_q = sum(1 for t in q_tokens if t <= 10) / len(q_tokens)
id_q = sum(1 for q in bundle.questions if any(ch.isupper() for ch in q.query.split()[-1])
           ) / len(bundle.questions)
multi_para = sum(1 for q in bundle.questions if q.hops >= 2) / len(bundle.questions)
corpus_tokens = sum(chunking.approx_tokens(d.body) for d in bundle.documents)

profile = {
    "has_structure": structured > 0.8,
    "answers_span_paragraphs": multi_para > 0.6,
    "factoid_queries": short_q > 0.4,
    "affordable_index_pass": corpus_tokens < 5_000_000,
}
tables.keyvalue([
    ("Documents with internal structure", f"{structured:.0%}  → has_structure = {profile['has_structure']}"),
    ("Questions spanning ≥2 documents", f"{multi_para:.0%}  → answers_span_paragraphs = {profile['answers_span_paragraphs']}"),
    ("Short questions (≤10 content tokens)", f"{short_q:.0%}  → factoid_queries = {profile['factoid_queries']}"),
    ("Corpus size", f"{corpus_tokens:,} tokens  → affordable_index_pass = {profile['affordable_index_pass']}"),
], title="Corpus profile — the inputs the tree needs", kicker="Measured, not assumed",
   caption="Every predicate below is derived from the corpus. That is the difference between "
           "running a decision tree and quoting one.")

In [ ]:
decision = catalog.CHUNKING_TREE.explain(profile)

In [ ]:
catalog.CHUNKING_TREE.show_table()

### Now check the tree against a measurement

Seven strategies, the same eval set, the same encoder, the same retrieval config. Index cost
and storage are measured rather than quoted.


In [ ]:
import time

def build_with(strategy, **kw):
    t0 = time.perf_counter()
    chunks = chunking.chunk_corpus(bundle.documents, strategy=strategy, **kw)
    em = embed.LsaEmbedder(dim=96).fit([d.title + "\n" + d.body for d in bundle.documents])
    vecs = em.encode_documents([c.text for c in chunks])
    idx = store.InMemoryIndex()
    idx.upsert(chunks, vecs, "v1", em.info.tag)
    build_ms = (time.perf_counter() - t0) * 1000
    p = pipeline.RagPipeline(idx, em, retrieve.RetrievalConfig(
        n_candidates=100, k=8, fusion="weighted", alpha=0.3, rerank="cross"), name=strategy)
    return chunks, p, build_ms

results = []
baseline_tokens = None
CHUNK_SIZE = 256          # held constant across strategies, so the comparison is about
                          # boundaries rather than about size
for strat in chunking.STRATEGIES:
    chunks, p, build_ms = build_with(strat, size_tokens=CHUNK_SIZE)
    st = chunking.chunk_stats(chunks)
    if strat == "fixed":
        baseline_tokens = st["total_tokens"]
    rs = pipeline.evaluate(p, eval_sample, chunks, personas=bundle.personas)
    s = metrics.summarize(rs)
    unresolved = sum(r["unresolved_gold"] for r in rs)
    results.append({
        "Strategy": strat,
        "Chunks": st["chunks"],
        "Median tokens": st["median_tokens"],
        "Storage ×": round(st["total_tokens"] / baseline_tokens, 2),
        "Index ms": round(build_ms),
        "Evidence recall": round(s["evidence_recall"], 3),
        "Full-chain": round(s["full_chain_recall"], 3),
        "Context precision": round(s["context_precision"], 3),
        "Unresolvable gold": unresolved,
    })

frame = pd.DataFrame(results).sort_values("Full-chain", ascending=False)
tables.show(frame, title="Seven chunking strategies on the same eval set",
            kicker="Measured",
            caption="Storage is relative to fixed-size chunking of the same corpus. "
                    "'Unresolvable gold' counts evidence spans that no chunk of this strategy "
                    "contains — a chunking choice can make a label unscoreable.",
            emphasize="Full-chain",
            highlight_rows=lambda r: r["Strategy"] == decision["outcome"].split(".")[0].lower()
                                     .replace(" chunking", "").strip())

In [ ]:
best = frame.iloc[0]["Strategy"]
tables.callout(
    f"The tree recommended <b>structural chunking</b> from the corpus profile. The "
    f"measurement puts <b>{best}</b> at the top on full-chain recall. Both are useful: the "
    "tree told you where to start and what to compare against, and the measurement told you "
    "what to ship."
    "<br><br>Read the storage and index-cost columns before you celebrate. Contextual "
    "chunking buys recall with index-time compute you pay again on every reindex, and "
    "late chunking inflates every embedded string. On a corpus that churns hourly those "
    "columns decide the question, not the recall column.", kind="note",
    title="Tree first, measurement second, cost column third")

In [ ]:
catalog.CHUNKING_MATRIX.show()

---

## 3.4 Query-time and the trace

The query path should record the retrieved items, their scores, the selected context, the
model response and the latency, so a failure can be reproduced. Candidates forget this in
interviews and regret it in production: **no trace, no debugging, and no eval set built from
real failures.**


In [ ]:
viz.flow(["Query", "Retrieve", "Re-rank", "Select top-k", "Answer", "Trace"],
         title="Query-time: assemble evidence for an answer",
         kicker="Section 3 · query path", highlight=5,
         caption="The last box is not an afterthought. It is what makes the other five "
                 "debuggable.", source="Deck slide 26")

In [ ]:
# The trace store is a queryable database, not a log file. That is the difference between
# "we log everything" and "show me every query where the gold chunk was retrieved and then
# dropped during packing".
for q in eval_sample[:40]:
    pipe.run(q.query, qid=q.qid, acl_groups=bundle.personas.get(q.persona))

print(f"traces stored: {pipe.trace_store.count()}\n")
df = pipe.trace_store.query('''
    SELECT qid,
           json_array_length(candidates) AS n_candidates,
           json_array_length(packed)     AS k_packed,
           ROUND(json_extract(stage_ms, '$.retrieve') +
                 json_extract(stage_ms, '$.rerank')   +
                 json_extract(stage_ms, '$.pack')     +
                 json_extract(stage_ms, '$.generate'), 1) AS total_ms,
           substr(query, 1, 46) AS question
    FROM traces ORDER BY total_ms DESC LIMIT 6
''')
print(df.to_string(index=False))

In [ ]:
# Diff two runs of the same query -- the "exceeds the bar" line of the build rubric.
subject = eval_sample[3]
a = pipe.run(subject.query, qid=subject.qid)
b = pipe.variant("k=4, no rerank", k=4, rerank="none").run(subject.query, qid=subject.qid)
d = trace.diff_traces(a, b)

gm, _ = metrics.resolve_gold(subject, pipe.chunks)
gold = {c for s in gm.values() for c in s}

print(f"QUERY  {d['query'][:78]}\n")
print(f"  candidates only in A            {len(d['candidates_only_in_a'])}")
print(f"  candidates only in B            {len(d['candidates_only_in_b'])}")
print(f"  packed only in A                {len(d['packed_only_in_a'])}")
print(f"  packed only in B                {len(d['packed_only_in_b'])}")
print(f"  retrieved then dropped, A       {len(d['retrieved_then_dropped_a'])}"
      f"   of which gold: {len(set(d['retrieved_then_dropped_a']) & gold)}")
print(f"  retrieved then dropped, B       {len(d['retrieved_then_dropped_b'])}"
      f"   of which gold: {len(set(d['retrieved_then_dropped_b']) & gold)}   ← the regression")
print(f"  latency delta                   {d['latency_delta_ms']:+.1f} ms")

"Retrieved then dropped, of which gold" is the row nobody instruments and everybody needs.
It separates *we could not find it* from *we found it and threw it away* — two failures with
completely different fixes, and only one of them is the retriever's fault.

---

## 3.5 Keeping an index fresh without a nightly rebuild

Two paths, and mixing them is the classic outage. The incremental path runs in minutes and is
triggered by content. The rebuild path runs in hours and is triggered by a *model* change — an
embedding model, a chunker, an analyzer — never by content.


In [ ]:
viz.hld([
    dict(name="Incremental path", note="minutes · triggered by content", tone="index", nodes=[
        ("1 · Change capture", "webhook or CDC emits document IDs, not documents"),
        ("2 · Content hash diff", "re-chunk only changed bodies; metadata-only edits skip "
                                  "embedding entirely"),
        ("3 · Chunk-level upsert", "stable IDs from doc_id + ordinal + hash"),
        ("4 · Soft-delete sweep", "tombstones stay filterable until compaction")]),
    dict(name="Rebuild path", note="hours to days · triggered by a model change", tone="warn",
         nodes=[
        ("Build v(n+1) beside v(n)", "both queryable, one routed to"),
        ("Shadow-evaluate", "frozen slice + replayed production queries"),
        ("Atomic alias swap", "keep v(n) warm"),
        ("Rollback", "a pointer change, not a rebuild")]),
], title="Keeping an index fresh without a nightly full rebuild", kicker="HLD",
   caption="Never write new-model embeddings into an index that still holds old-model "
           "vectors. Vectors from different models are not comparable, and cosine similarity "
           "will not tell you they are wrong.",
   source="Deck slide 27")

In [ ]:
# ── The incremental path, executed ──────────────────────────────────────────
target = bundle.documents[0]
before_hash = target.content_hash

# (a) a metadata-only edit: no re-embedding at all
meta_only = corpus.Document(target.doc_id, target.title, target.source, "2023-08-20",
                            target.passages, target.acl, target.tenant, target.entities)
print("CHANGE 1 — publication date corrected (metadata only)")
print(f"  body hash {before_hash} → {meta_only.content_hash}   "
      f"{'unchanged → skip embedding entirely' if meta_only.content_hash == before_hash else 'changed'}")

# (b) a real body edit: re-chunk and upsert only this document
edited_passages = list(target.passages)
edited_passages[2] = corpus.Passage(
    edited_passages[2].heading,
    edited_passages[2].text + " A correction was issued the following week.",
    edited_passages[2].facts, edited_passages[2].anchor)
edited = corpus.Document(target.doc_id, target.title, target.source, target.published,
                         tuple(edited_passages), target.acl, target.tenant, target.entities)

old_chunks = [c for c in pipe.chunks if c.doc_id == target.doc_id]
new_chunks = chunking.chunk_corpus([edited], strategy="structural")
old_ids = {c.chunk_id for c in old_chunks}
new_ids = {c.chunk_id for c in new_chunks}

print("\nCHANGE 2 — one passage edited (body changed)")
print(f"  body hash {before_hash} → {edited.content_hash}")
print(f"  chunks before {len(old_chunks)}   after {len(new_chunks)}")
print(f"  stable (id unchanged, no re-embed needed)  {len(old_ids & new_ids)}")
print(f"  new or changed (must be embedded)          {len(new_ids - old_ids)}")
print(f"  orphaned (must be tombstoned)              {len(old_ids - new_ids)}")

index.upsert(new_chunks, pipe.embedder.encode_documents([c.text for c in new_chunks]),
             "v1", pipe.embedder.info.tag)
index.tombstone(sorted(old_ids - new_ids), "v1")
print(f"\n  after upsert + tombstone: {index.stats('v1')}")

In [ ]:
tables.callout(
    "<b>Stable chunk IDs are the whole trick.</b> The id is <code>doc_id + ordinal + content "
    "hash</code>, so an unchanged chunk keeps its id and needs no new vector, and a changed "
    "one gets a new id that an upsert can write over. Delete-then-insert orphans rows; "
    "upsert-then-tombstone does not."
    "<br><br>Tombstoned chunks stay filterable until the next compaction, so a query already "
    "in flight keeps reading a consistent index instead of watching rows vanish underneath "
    "it.", kind="note", title="Why the id is not an implementation detail")

In [ ]:
# ── The rebuild path, and the outage it prevents ────────────────────────────
# Build v2 with a different encoder, alongside v1, both queryable.
em_v2 = embed.LsaEmbedder(dim=48, version="2.0").fit(
    [d.title + "\n" + d.body for d in bundle.documents])
index.upsert(pipe.chunks, em_v2.encode_documents([c.text for c in pipe.chunks]),
             "v2", em_v2.info.tag)

print("BLUE / GREEN")
print(f"  v1  {index.stats('v1')['chunks']} chunks  encoder {pipe.embedder.info.tag}")
print(f"  v2  {index.stats('v2')['chunks']} chunks  encoder {em_v2.info.tag}")
print(f"  alias 'live' currently points at: {index.resolve('live')}\n")

# Shadow-evaluate v2 on the frozen slice before any traffic moves.
frozen = [q for q in bundle.questions if q.slice == "frozen" and q.question_type != "null"]
shadow = {}
for version, em in (("v1", pipe.embedder), ("v2", em_v2)):
    p = pipeline.RagPipeline(index, em, retrieve.RetrievalConfig(
        n_candidates=100, k=8, fusion="weighted", alpha=0.3, rerank="cross",
        index_version=version), name=version)
    rs = pipeline.evaluate(p, frozen, pipe.chunks, personas=bundle.personas)
    shadow[version] = metrics.summarize(rs)

print(f"SHADOW EVALUATION on the frozen slice ({len(frozen)} questions)")
for v, s in shadow.items():
    print(f"  {v}  evidence recall {s['evidence_recall']:.3f}   "
          f"full-chain {s['full_chain_recall']:.3f}")
delta = shadow["v2"]["full_chain_recall"] - shadow["v1"]["full_chain_recall"]
print(f"\n  delta {delta:+.3f} → " +
      ("swap the alias" if delta >= 0 else "do not swap — v1 stays live, and it cost you nothing"))

In [ ]:
# ── The outage: mixing the two paths ────────────────────────────────────────
# Write a handful of v2 (48-dim) vectors into the v1 index and watch what happens.
# The dangerous case is not a dimension mismatch -- that at least raises. It is a *new
# encoder of the same dimension*, whose vectors slot into the existing schema perfectly and
# mean something entirely different. Here: the same LSA at the same 96 dimensions, fitted
# on chunk text instead of document text. Same shape, different latent space.
em_same_dim = embed.LsaEmbedder(dim=96, version="1.1-refit").fit(
    [c.text for c in pipe.chunks])

poisoned = store.InMemoryIndex()
poisoned.upsert(pipe.chunks, pipe.embedder.encode_documents([c.text for c in pipe.chunks]),
                "v1", pipe.embedder.info.tag)
check_before = poisoned.mixed_version_check("v1")

sub = pipe.chunks[: len(pipe.chunks) // 3]        # a third of the index, as a partial reindex
poisoned.upsert(sub, em_same_dim.encode_documents([c.text for c in sub]),
                "v1", em_same_dim.info.tag)
check_after = poisoned.mixed_version_check("v1")

p_bad = pipeline.RagPipeline(poisoned, pipe.embedder, retrieve.RetrievalConfig(
    n_candidates=100, k=8, fusion="dense", rerank="none"), name="poisoned")
p_ok = pipeline.RagPipeline(index, pipe.embedder, retrieve.RetrievalConfig(
    n_candidates=100, k=8, fusion="dense", rerank="none", index_version="v1"), name="clean")

bad = metrics.summarize(pipeline.evaluate(p_bad, eval_sample[:40], pipe.chunks,
                                          personas=bundle.personas))
ok = metrics.summarize(pipeline.evaluate(p_ok, eval_sample[:40], pipe.chunks,
                                         personas=bundle.personas))

print("MIXED-VERSION INDEX")
print(f"  before: {check_before}")
print(f"  after:  {check_after}")
print(f"\n  a partial reindex wrote {len(sub)} of {len(pipe.chunks)} rows with the new encoder")
print(f"  dense evidence recall   clean {ok['evidence_recall']:.3f}   "
      f"poisoned {bad['evidence_recall']:.3f}   ({bad['evidence_recall']-ok['evidence_recall']:+.3f})")
print("\n  No exception was raised. No dimension mismatch. Cosine similarity returned")
print("  perfectly well-formed numbers for vectors that mean nothing to each other.")
print("  The only thing that caught it is the model-version tag on the row.")

That is the outage in full. Nothing crashed, no alert fired, and the system returned confident
scores for comparisons that were meaningless. The `embedder_tag` column — one string per row —
is the entire defence, and `mixed_version_check()` belongs in your release gate, not in your
incident review.

---

## 3.6 Where the retrieval state should live

The real question is rarely *which vector database*. It is whether you need one at all
at their scale, and what the answer costs them in operations.


In [ ]:
catalog.STATE_LOCATION.show()

In [ ]:
from raglab.trees import score_matrix

# A concrete client scenario, scored rather than argued about.
weights = {"Data residency / compliance fit": 5, "Time to first demo": 4,
           "Operational headcount required": 4, "Scale headroom": 3,
           "Cost predictability": 3, "One ACL model for lexical + vectors": 4}
scores = {
    "In-process (FAISS)":      {"Data residency / compliance fit": 5, "Time to first demo": 5,
                                "Operational headcount required": 5, "Scale headroom": 1,
                                "Cost predictability": 5,
                                "One ACL model for lexical + vectors": 2},
    "Postgres + pgvector":     {"Data residency / compliance fit": 5, "Time to first demo": 4,
                                "Operational headcount required": 3, "Scale headroom": 3,
                                "Cost predictability": 4,
                                "One ACL model for lexical + vectors": 3},
    "Search engine (OpenSearch)": {"Data residency / compliance fit": 4, "Time to first demo": 2,
                                "Operational headcount required": 1, "Scale headroom": 5,
                                "Cost predictability": 3,
                                "One ACL model for lexical + vectors": 5},
    "Managed vector DB":       {"Data residency / compliance fit": 2, "Time to first demo": 5,
                                "Operational headcount required": 5, "Scale headroom": 4,
                                "Cost predictability": 2,
                                "One ACL model for lexical + vectors": 2},
}
ranked = score_matrix(
    catalog.STATE_LOCATION, weights, scores,
    title="Scenario: regulated client, 8M chunks, two-person delivery team, data must not "
          "leave their VPC")

In [ ]:
tables.callout(
    "The weights are the argument, not the scores. Anyone can rank four options; what a panel "
    "is listening for is whether you made the criteria explicit and weighted them <i>before</i> "
    "you looked at the answer. In a regulated environment the decision is usually settled by "
    "data residency and the existing ACL model long before any recall benchmark — so ask about "
    "both in discovery, before you benchmark anything.",
    kind="interview", title="What this table is really for")

---

## 3.7 Permission-aware retrieval

If you take one enterprise lesson from this curriculum, take this one. **Post-filtering is
the leak.** It fails in two ways, both silent.

### What the code is about to do

Run the same query as four personas under both filter designs, and measure the two failures:
`k` collapsing, and restricted documents influencing the ranking of everything around them.


In [ ]:
viz.hld([
    dict(name="Post-filter · wrong", tone="warn", nodes=[
        ("Retrieve top-k globally", "the restricted chunk is a candidate"),
        ("Rank", "and it influences every neighbour's rank"),
        ("Drop what the user may not see", "k collapses — two chunks instead of eight"),
        ("Answer", "from whatever survived")]),
    dict(name="Pre-filter · right", tone="control", nodes=[
        ("Push the ACL predicate into the search", "the candidate pool is already scoped"),
        ("Rank", "restricted chunks were never candidates"),
        ("Pack k", "full k, every time"),
        ("Answer", "provably uninfluenced")]),
], title="Permission-aware retrieval: where the filter goes", kicker="Enterprise reality",
   caption="Post-filtering satisfies 'the user cannot read it'. It does not satisfy 'no "
           "answer may be influenced by it' — and that is the sentence legal actually writes.",
   source="Deck slide 29")

In [ ]:
probe_q = "What is the recommended fix for ERR_CONN_RESET?"
rowsp = []
for persona, groups in bundle.personas.items():
    for mode in ("pre", "post"):
        v = pipe.variant(f"{persona}/{mode}", filter_mode=mode)
        tr = v.run(probe_q, acl_groups=groups)
        rowsp.append([persona, mode, len(tr.packed), tr.k_collapse,
                      sum(b["tokens"] for b in tr.packed),
                      "abstained" if metrics.abstained(tr.answer) else tr.answer[:44]])

tables.show(pd.DataFrame(rowsp, columns=["Persona", "Filter", "Chunks packed",
                                         "k collapse", "Evidence tokens", "Answer"]),
            title=f"One query, four personas, two filter designs",
            kicker="Measured · " + probe_q,
            caption="Post-filtering does not just hide restricted evidence — it silently "
                    "shrinks the context for exactly the users with the narrowest access, "
                    "who then get the worst answers and no explanation.",
            emphasize="k collapse",
            highlight_rows=lambda r: r["k collapse"] > 0)

In [ ]:
# The second failure: restricted documents influencing the ranking of what remains.
allowed_only = pipe.variant("pre", filter_mode="pre").run(
    probe_q, acl_groups=bundle.personas["analyst"])
global_then_drop = pipe.variant("post", filter_mode="post").run(
    probe_q, acl_groups=bundle.personas["analyst"])
unrestricted = pipe.variant("none", filter_mode="pre").run(probe_q, acl_groups=None)

print("SCORE LEAK — does a restricted document change the ranking a permitted user sees?")
print(f"  unscoped search returns   {len(unrestricted.packed)} chunks, "
      f"{sum(1 for b in unrestricted.packed if 'in-' in b['doc_id'] or 'sp-' in b['doc_id'])} "
      "of them restricted")
print(f"  post-filter leaves        {len(global_then_drop.packed)} chunks for the analyst")
print(f"  pre-filter returns        {len(allowed_only.packed)} chunks for the analyst\n")

pre_ids = [b["chunk_id"] for b in allowed_only.packed]
post_ids = [b["chunk_id"] for b in global_then_drop.packed]
print(f"  the two permitted sets are {'IDENTICAL' if pre_ids == post_ids else 'DIFFERENT'}")
print("  Under post-filtering the restricted chunks occupied ranks that shifted every")
print("  permitted chunk beneath them, and the count and latency of the response are")
print("  themselves observable — the existence of restricted documents is inferable.")

In [ ]:
# The test that belongs in your release gate, not in a one-off review.
def assert_persona_isolation(pipeline_obj, questions, personas, restricted_prefixes=("lg-", "fi-", "sp-", "in-")):
    '''Run the same queries as two personas and assert disjoint restricted evidence.

    This is interview question Q4's "prove it" bullet, written down. It runs in the release
    gate: a change to filters, namespaces, chunking or the ACL denormalisation that breaks
    isolation fails the build rather than reaching UAT.
    '''
    failures = []
    for q in questions:
        for name, groups in personas.items():
            tr = pipeline_obj.variant("gate", filter_mode="pre").run(q, acl_groups=groups)
            for b in tr.packed:
                row = pipeline_obj.index.get(b["chunk_id"])
                acl = set(__import__("json").loads(row["acl"]))
                if not acl & set(groups):
                    failures.append((name, q[:40], b["chunk_id"], sorted(acl)))
    return failures

probe_questions = [probe_q,
                   "What risk did the integration review identify for Northwind Systems?",
                   "How far below the announced figure are the modelled synergies?",
                   "What did the failover drill find about client reconnection behaviour?"]
viol = assert_persona_isolation(pipe, probe_questions, bundle.personas)
print(f"persona isolation test: {len(probe_questions)} queries × {len(bundle.personas)} personas")
print(f"  violations: {len(viol)}")
assert not viol, viol[:3]
print("  PASS — no persona ever received a chunk outside its groups.")

In [ ]:
tables.show(pd.DataFrame([
    ["Where does the ACL live at query time?",
     "Denormalised onto the chunk (fast, needs its own change stream) or resolved per request "
     "from the source system (always current, adds latency to every query)."],
    ["How fast must a revocation propagate?",
     "Minutes means denormalised ACLs need their own CDC stream. State the propagation lag as "
     "a number — legal will ask, and 'quickly' is not an answer."],
    ["Namespace or filter?",
     "Namespaces isolate cleanly and cost you cross-tenant recall. Filters share an index and "
     "cost you tuning — and a selective filter degrades graph traversal (notebook 04)."],
    ["Is the cache tenant-scoped?",
     "A shared prompt-prefix cache across tenants is a data-leak class of bug, not a "
     "performance issue. Notebook 07 keys the cache correctly."],
    ["Do traces store retrieved text?",
     "If yes, the trace store inherits the corpus's compliance boundary — including its "
     "retention policy and its jurisdiction."],
], columns=["Design question", "What you have to be able to answer"]),
    title="The design checklist legal will walk you through",
    kicker="Enterprise reality",
    caption="Every row is a place where a system that passes a functional test still leaks. "
            "Ask all five in discovery.",
    source="Deck slide 29", emphasize="Design question")

---

## 3.8 Failure points, and a case study

### The four section-3 signatures


In [ ]:
fp = []
# Boundary loss: premise and conclusion split with no overlap.
no_overlap = chunking.chunk_corpus(bundle.documents, strategy="fixed", overlap=0.0)
with_overlap = chunking.chunk_corpus(bundle.documents, strategy="fixed", overlap=0.25)
def resolvable(chunks):
    return sum(len(metrics.resolve_gold(q, chunks)[1]) for q in eval_sample)
fp.append(["Boundary loss", "fixed chunking with 0% vs 25% overlap",
           f"{resolvable(no_overlap)} vs {resolvable(with_overlap)} gold spans that no chunk "
           "contains",
           "A premise and its conclusion split across chunks. The label becomes unscoreable "
           "and the answer unretrievable — at once."])
# Stale index
fp.append(["Stale index", "one document edited, index not updated",
           f"content hash {before_hash} → {edited.content_hash} with the old vector still "
           "live",
           "Detected by a content-hash diff in the incremental path. Undetectable by any "
           "query-time metric."])
# Metadata drop
no_dates = sum(1 for c in pipe.chunks if not c.published)
fp.append(["Metadata drop", "publication date missing from the index",
           f"{no_dates} chunks affected here; every temporal filter would silently return "
           "nothing",
           "A field you did not index is a filter you cannot apply — and the failure looks "
           "like poor recall, not like a schema bug."])
# Missing trace
fp.append(["Missing trace", "a bad answer with no stored candidates or scores",
           f"trace store holds {pipe.trace_store.count()} replayable runs",
           "Without it you cannot reproduce the failure, cannot diff two runs, and cannot "
           "turn a production failure into a regression case."])

tables.show(pd.DataFrame(fp, columns=["Signature", "Reproduced by", "Measured here",
                                      "Why it is hard to see"]),
            title="Failure points: RAG system design", kicker="Failure points",
            caption="Stale index and missing trace are the two that reach the client. The "
                    "other two you will catch yourself, if you are looking.",
            source="Deck slide 30", emphasize="Signature")

### Case study — LinkedIn, knowledge-graph RAG for customer support (2024)

Historical support tickets are structured objects: issue, environment, steps tried,
resolution, linked tickets. Split into flat text chunks, a retrieved fragment could return the
symptom while leaving the resolution in a different chunk that never scored high enough. They
parsed tickets into a knowledge graph whose nodes preserve ticket sections and whose edges
preserve relationships between tickets, and retrieved over sub-graphs — so the unit returned
to the model is a coherent case, not a fragment. Reported outcome: **−28.6% median per-issue
resolution time**, alongside gains in retrieval and answer quality.

The lesson is representation, not retrieval algorithm. Our corpus has the same shape: an
incident report and its customer-facing KB article are one case split across two documents.


In [ ]:
# Retrieve the fragment, or retrieve the case? Group each incident with its KB article and
# treat the pair as one retrievable unit.
case_docs = []
for d in bundle.documents:
    if d.source not in ("incident", "support_kb"):
        case_docs.append(d)
grouped = {}
for d in bundle.documents:
    if d.source in ("incident", "support_kb"):
        key = tuple(sorted(e for e in d.entities))[:1] or (d.doc_id,)
        grouped.setdefault(key, []).append(d)
for key, docs in grouped.items():
    if len(docs) == 1:
        case_docs.append(docs[0]); continue
    passages = tuple(p for d in sorted(docs, key=lambda x: x.doc_id) for p in d.passages)
    case_docs.append(corpus.Document(
        "case-" + docs[0].doc_id, " / ".join(d.title for d in docs), "incident",
        min(d.published for d in docs), passages,
        tuple(sorted({g for d in docs for g in d.acl})), docs[0].tenant, docs[0].entities))

support_qs = [q for q in bundle.questions if q.persona == "support_engineer"][:24]
runs = {}
for name, docs in (("flat chunks (fragment)", bundle.documents),
                   ("case-level unit (graph-like)", case_docs)):
    ch, p, _ = build_with("structural") if docs is bundle.documents else (None, None, None)
    if docs is not bundle.documents:
        ch = chunking.chunk_corpus(docs, strategy="structural")
        em = embed.LsaEmbedder(dim=96).fit([d.title + "\n" + d.body for d in docs])
        idx = store.InMemoryIndex()
        idx.upsert(ch, em.encode_documents([c.text for c in ch]), "v1", em.info.tag)
        p = pipeline.RagPipeline(idx, em, retrieve.RetrievalConfig(
            n_candidates=100, k=8, fusion="weighted", alpha=0.3, rerank="cross"), name=name)
    runs[name] = pipeline.evaluate(p, support_qs, ch, personas=bundle.personas)

print(pipeline.compare_runs(runs, keys=("evidence_recall", "full_chain_recall",
                                        "context_precision")).to_string(index=False))

In [ ]:
tables.callout(
    "<b>Before tuning a retriever, ask what the natural retrievable unit of this corpus is.</b> "
    "For tickets it is a case. For contracts it is a clause. For code it is a symbol and its "
    "call sites. Our incident-plus-KB pairing is a crude version of what LinkedIn built, and "
    "it moves full-chain recall on exactly the questions that need both halves."
    "<br><br>The cost is real and worth naming out loud: a parser per source system, a graph "
    "store to operate, and a schema that has to be maintained as the ticket template changes. "
    "This is worth it when the corpus is highly structured and high volume — and rarely "
    "otherwise.", kind="note", title="The engineering read")

---

## 3.9 The interview


In [ ]:
tables.show(pd.DataFrame([
    [catalog.SECTION_QUESTIONS[3][0],
     "Whether you separate the incremental path from the rebuild path",
     "Change capture emits IDs, content-hash diff decides what re-embeds, chunk-level upsert "
     "on stable ids, tombstone sweep. Rebuilds are triggered by model changes only, run "
     "blue/green, are shadow-evaluated on the frozen slice, and swap by alias. Say 'versioned "
     "index' and 'blue-green swap' out loud."],
    [catalog.SECTION_QUESTIONS[3][1],
     "Whether you know a field must be indexed to be filterable",
     "In the index: anything you filter or scope on — source, date, tenant, ACL. In the "
     "prompt: anything the model must reason about — the date, the title, the source. Dates "
     "go in both, which is the answer most candidates miss."],
    [catalog.SECTION_QUESTIONS[3][2],
     "Whether you answer with a number or with a procedure",
     "The shape of the document and the shape of the question, then measure. Quote the "
     "measurement you would run: seven strategies, one eval set, and the storage and "
     "index-cost columns next to the recall column."],
    [catalog.SECTION_QUESTIONS[3][3],
     "Whether you have ever had to reproduce a failure at 2am",
     "Retrieved ids and scores per stage, the packed context with provenance, the prompt, the "
     "response, per-stage latency, the index version and the encoder tag. Enough to diff two "
     "runs of one query and see 'retrieved then dropped'."],
], columns=["Question", "What the panel is testing", "What a strong answer covers"]),
    title="Typical interview questions: RAG system design",
    kicker="Section 3 · interview", source="Deck slide 32", emphasize="Question")

---

## 3.10 Checkpoint

1. A client's documents change daily but their embedding model has not changed in a year.
   Which path runs, and how often?
2. You add an ACL column to the chunk table and start pre-filtering. What did you just do to
   your ANN recall, and how would you find out?
3. Your chunking change improved evidence recall by 4 points. Name two columns you must look
   at before shipping it.


In [ ]:
print("1 ·  The incremental path only, on the source's change signal — typically minutes.")
print("     The rebuild path is triggered by an embedding-model, chunker or analyzer change,")
print("     never by content. A nightly full rebuild for daily content changes is paying")
print("     rebuild cost for an incremental problem.\n")

print("2 ·  You pushed a predicate into the graph traversal, and a selective filter can strand")
print("     it in a sparse region. Find out by measuring ANN recall against flat search WITH")
print("     the real filters on — notebook 04 does exactly that and the gap is large.\n")

print("3 ·  Storage multiplier and index cost. From the table above:")
sub = frame[["Strategy", "Storage ×", "Index ms", "Evidence recall", "Full-chain"]]
print(sub.to_string(index=False))
print("\n     A 4-point recall gain bought with 2.4x storage and a per-reindex compute bill is")
print("     a trade, not a win — and on a corpus that churns hourly it can be a bad one.")

---

## What carries forward

- Four planes, two SLAs. Index-time is a deploy; query-time is a p95; the control plane is
  how you change either one deliberately.
- Chunking is chosen from the corpus profile and settled by measurement — with the storage
  and index-cost columns in the same table as recall.
- Stable chunk ids, content-hash diffs, tombstones and an atomic alias swap are what make
  freshness cheap. The `embedder_tag` is what stops the silent outage.
- Pre-filter, always. Post-filtering collapses `k` for your most restricted users and leaks
  through ranks, counts and latency. The two-persona isolation test belongs in the gate.

**Next:** `04_retrieval_methods_and_reranking.ipynb` — BM25 from first principles, the
analyzer trap that makes identifiers unsearchable, ANN recall curves, fusion that actually
earns its place, and a reranker you fit and then verify on a slice it never saw.
